# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [2,4,6] -> [1, 2, 3] [0, 1, 1] -> операция происходит с каждым элементом
2. [[1,1,1,1],[1,1,1,1],[1,1,1,1]] -> [4, 4, 4] -> asix=1 значит схлопываем по столбцам
3. [0,1,2]+[[0],[1],[2]] -> broadcast -> [[0,1,2],[1,2,3],[2,3,4]] -> вектор бродкастится на матрицу
4. [1,4,9] -> ([1,2,3]) -> 6 -> srt() это квадратный корень из числа, операция происходит с каждым элементом
5. [1,2,3,4] -> [1,4,9,16] -> (30/4) -> 7.5 -> каждый элемент возводится в квадрат -> среднее арифметическое

In [192]:
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [193]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [194]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [195]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [196]:
# 1 Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?
oct_amt = amount[(clean) & (month == '2026-10')].sum()
sep_amt = amount[(clean) & (month == '2026-09')].sum()
print(clean.sum())
print(oct_amt, sep_amt)
print(round(((oct_amt - sep_amt)/sep_amt * 100),2))

4739
34419520.0 35243410.0
-2.34


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [197]:
# 2 Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
# Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?
categories = np.unique(cat)
months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table_1 = np.array([[amount[(clean) & (month == m) & (cat == c)].sum() for m in months] for c in categories])

for c, r in zip(categories, table_1):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")

Дом           4,997,730    5,125,790
Книги         1,065,300    1,140,910
Красота       2,610,950    2,678,990
Одежда        9,167,100    8,505,730
Электроника   17,402,330   16,968,100


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [198]:
# 3 Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
# Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?
table_2 = np.array([[sum((clean) & (month == m) & (cat == c)) for m in months] for c in categories])

for c, amt, cnt in zip(categories, table_1, table_2):
    print(f"{c:<12} {amt[0]/cnt[0]:>12,.0f} {amt[1]/cnt[1]:>12,.0f}")

Дом                10,499       10,768
Книги               4,020        3,816
Красота             6,187        6,159
Одежда             13,521       13,046
Электроника        32,773       33,600


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [199]:
# 4 Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
# Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])

prc = table / table.sum(axis=0)

print('Сумма столбцов', prc.sum(axis=0))

for c, p in zip(cities, prc):
    print(f"{c:<10} {p[0]:>12,.4f} {p[1]:>12,.4f}")

print('Ответ: По октябрю доля заказов из Шымкента снизилась на', round(prc[5, 0] - prc[5, 1], 4))

Сумма столбцов [1. 1.]
Актобе           0.0731       0.0738
Алматы           0.4115       0.4254
Астана           0.2563       0.2571
Атырау           0.0262       0.0282
Караганда        0.1035       0.0980
Шымкент          0.1293       0.1175
Ответ: По октябрю доля заказов из Шымкента снизилась на 0.0119


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [200]:
# 5 Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
# Напишите менеджеру одну фразу: где искать причину.
prc_city_diff_val = (table[:,0] - table[:,1]) / (sep_amt - oct_amt) * 100
#print(prc_city_diff_val)
# нагляднее
for c, r in zip(cities, prc_city_diff_val):
    print(f"{c:<10} {r:>12,.4f}")

print(prc_city_diff_val.sum()) #проверка процента

Актобе           4.4351
Алматы         -17.2292
Астана          22.2955
Атырау          -5.5675
Караганда       33.6016
Шымкент         62.4646
100.0


Ответ: берем все города из этой таблицы от большей к меньшей и копаем дальше. Самая плохая ситуация в Шымкенте, из имеющихся полей можно проверить, как изменилась доставка. Возможно упало качество сервиса.

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [201]:
# 6 AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
#```python
#share = table / table.sum(axis=1)
#```

reshaped = table.sum(axis=1, keepdims=True).reshape(6,1)

print(table, reshaped, sep='\n')

share = table / reshaped
print(share.sum(axis=1)) #проверка долей

print("Ответ: ", share, sep='\n')


[[ 2576960.  2540420.]
 [14501070. 14643020.]
 [ 9034220.  8850530.]
 [  923990.   969860.]
 [ 3648520.  3371680.]
 [ 4558650.  4044010.]]
[[ 5117380.]
 [29144090.]
 [17884750.]
 [ 1893850.]
 [ 7020200.]
 [ 8602660.]]
[1. 1. 1. 1. 1. 1.]
Ответ: 
[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]


Ответ: broadcast не происходил потому что не совпадали формы массивов. Вектор с суммами нужно было изменить на массив (6,1) чтобы к каждой строке из массива (6,2) была присоединена строка.

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [202]:
# 7 Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
# Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.
clean_paid = (clean) & (paid)

amt_clean_paid = amount[clean_paid]
z_score = (amt_clean_paid - amt_clean_paid.mean()) / amt_clean_paid.std()
print(-1 < z_score.mean() < 1)
print(z_score.std())

z_abs_bigger_3 = (abs(z_score) > 3).sum()

print('Сколько заказов с |z| > 3? Ответ: ', z_abs_bigger_3)
print('Какая это доля? Ответ: ',round(z_abs_bigger_3/len(amt_clean_paid) * 100, 2))


True
1.0
Сколько заказов с |z| > 3? Ответ:  87
Какая это доля? Ответ:  1.84


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [203]:
# 8 Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
# Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
# Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

# amt days
clean_not_nan_days = (~np.isnan(days)) & (paid)
amt_vec = amount[clean_not_nan_days].reshape(-1)
days_vec = days[clean_not_nan_days].reshape(-1)

days_x_amt = np.column_stack((amt_vec, days_vec))

result = (days_x_amt - days_x_amt.min(axis=0)) / (days_x_amt.max(axis=0) - days_x_amt.min(axis=0))

print(result.min(axis=0), result.max(axis=0))
print(result)

[0. 0.] [1. 1.]
[[0.00073505 0.        ]
 [0.00029102 0.14285714]
 [0.00084006 0.28571429]
 ...
 [0.00107108 0.14285714]
 [0.00029202 0.14285714]
 [0.0014191  0.14285714]]


Ответ: чтобы избавиться от перекоса признаков с разными единицами исчисления. Здесь мы сравниваем сумму денег(0-999999...) и кол-во дней(0-4). Если оставить все как есть, то модель преувеличит значимость признака.